Кредитный скоринг

1. Анализ данных

In [54]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

%matplotlib inline

In [55]:
train_df = pd.read_csv('/content/credit_scoring_train.csv', index_col='client_id')
test_df = pd.read_csv('/content/credit_scoring_test.csv', index_col='client_id')
print(train_df[:10])

                   DIR   Age  NumLoans  NumRealEstateLoans  NumDependents  \
client_id                                                                   
0             0.496289  49.1        13                   0            0.0   
1             0.433567  48.0         9                   2            2.0   
2          2206.731199  55.5        21                   1            NaN   
3           886.132793  55.3         3                   0            0.0   
4             0.000000  52.3         1                   0            0.0   
5             0.121965  73.5        18                   1            1.0   
6             0.461193  71.3         9                   0            0.0   
7             2.624982  46.3         6                   0            0.0   
8             0.074518  28.1        10                   0            0.0   
9             0.266679  31.3         6                   0            0.0   

           Num30-59Delinquencies  Num60-89Delinquencies        Income  \
cl

In [56]:
print(test_df[:10])

                   DIR   Age  NumLoans  NumRealEstateLoans  NumDependents  \
client_id                                                                   
75000         0.488558  39.2         7                   2            2.0   
75001         0.132810  42.3         8                   0            1.0   
75002      1784.812905  51.5         5                   1            0.0   
75003         0.538571  57.1        30                   2            0.0   
75004         0.098539  70.1         3                   0            0.0   
75005         0.014285  57.0         6                   0            0.0   
75006         0.133529  56.5         9                   1            0.0   
75007         0.426060  38.2         8                   1            2.0   
75008         0.090592  44.3         9                   0            2.0   
75009       590.142422  39.3         4                   0            2.0   

           Num30-59Delinquencies  Num60-89Delinquencies        Income  \
cl

In [57]:
y = train_df['Delinquent90']
train_df.drop('Delinquent90', axis=1, inplace=True)

In [58]:
train_df.head()

,DIR,Age,NumLoans,NumRealEstateLoans,NumDependents,Num30-59Delinquencies,Num60-89Delinquencies,Income,BalanceToCreditLimit
client_id,,,,,,,,,
0,0.496289,49.1,13,0,0.0,2,0,5298.360639,0.387028
1,0.433567,48.0,9,2,2.0,1,0,6008.056256,0.234679
2,2206.731199,55.5,21,1,NaN,1,0,NaN,0.348227
3,886.132793,55.3,3,0,0.0,0,0,NaN,0.971930
4,0.000000,52.3,1,0,0.0,0,0,2504.613105,1.004350


Посмотрим на число пропусков в каждом признаке.

In [59]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 75000 entries, 0 to 74999
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   DIR                    75000 non-null  float64
 1   Age                    75000 non-null  float64
 2   NumLoans               75000 non-null  int64  
 3   NumRealEstateLoans     75000 non-null  int64  
 4   NumDependents          73084 non-null  float64
 5   Num30-59Delinquencies  75000 non-null  int64  
 6   Num60-89Delinquencies  75000 non-null  int64  
 7   Income                 60153 non-null  float64
 8   BalanceToCreditLimit   75000 non-null  float64
dtypes: float64(5), int64(4)
memory usage: 5.7 MB


In [60]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 75000 entries, 75000 to 149999
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   DIR                    75000 non-null  float64
 1   Age                    75000 non-null  float64
 2   NumLoans               75000 non-null  int64  
 3   NumRealEstateLoans     75000 non-null  int64  
 4   NumDependents          72992 non-null  float64
 5   Num30-59Delinquencies  75000 non-null  int64  
 6   Num60-89Delinquencies  75000 non-null  int64  
 7   Income                 60116 non-null  float64
 8   BalanceToCreditLimit   75000 non-null  float64
dtypes: float64(5), int64(4)
memory usage: 5.7 MB


Заменим пропуски средними значениями.

In [61]:
train_df['NumDependents'].fillna(train_df['NumDependents'].mean(), inplace=True)
train_df['Income'].fillna(train_df['Income'].mean(), inplace=True)
test_df['NumDependents'].fillna(test_df['NumDependents'].mean(), inplace=True)
test_df['Income'].fillna(test_df['Income'].mean(), inplace=True)

/tmp/ipykernel_729/610343124.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['NumDependents'].fillna(train_df['NumDependents'].mean(), inplace=True)
/tmp/ipykernel_729/610343124.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method

Дерево решений без настройки параметров - 0.53583

In [62]:
first_tree =  DecisionTreeClassifier(max_depth=3, random_state=17)
first_tree.fit(train_df, y)

DecisionTreeClassifier(max_depth=3, random_state=17)

In [63]:
first_tree_pred = first_tree.predict(test_df)

In [64]:
def write_to_submission_file(predicted_labels, out_file,
                             target='Delinquent90', index_label="client_id"):
    # turn predictions into data frame and save as csv file
    predicted_df = pd.DataFrame(predicted_labels,
                                index = np.arange(75000,
                                                  predicted_labels.shape[0] + 75000),
                                columns=[target])
    predicted_df.to_csv(out_file, index_label=index_label)

In [65]:
write_to_submission_file(first_tree_pred, 'credit_scoring_first_tree.csv')

Предсказания вероятности дефолта -  0.80525



In [66]:
first_tree_pred_probs = first_tree.predict_proba(test_df)[:, 1]
write_to_submission_file(first_tree_pred_probs, 'credit_scoring_first_tree_probs.csv')

Дерево решений с настройкой параметров с помощью GridSearch - 0.83638

In [67]:
tree_params = {'max_depth': list(range(3, 8)),
               'min_samples_leaf': list(range(5, 13))}

locally_best_tree = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=17),
    param_grid=tree_params,
    cv=5,
    n_jobs=-1,
    scoring='roc_auc'
)
locally_best_tree.fit(train_df, y)

print("Лучшие параметры:", locally_best_tree.best_params_)
print("Лучший ROC-AUC на кросс-валидации:", round(locally_best_tree.best_score_, 3))

Лучшие параметры: {'max_depth': 6, 'min_samples_leaf': 12}
Лучший ROC-AUC на кросс-валидации: 0.829


In [68]:
tuned_tree_pred_probs = locally_best_tree.predict_proba(test_df)[:, 1]
write_to_submission_file(tuned_tree_pred_probs, 'credit_scoring_tuned_tree.csv')

Случайный лес без настройки параметров - 0.82388


In [69]:
first_forest = RandomForestClassifier(random_state=17, n_jobs=-1)
first_forest.fit(train_df, y)

first_forest_pred_probs = first_forest.predict_proba(test_df)[:, 1]
write_to_submission_file(first_forest_pred_probs, 'credit_scoring_first_forest.csv')

Случайный лес c настройкой параметров - 0.82278


In [70]:
%%time
forest_params = {'max_features': np.linspace(.3, 1, 7)}

locally_best_forest = GridSearchCV(
    estimator=RandomForestClassifier(random_state=17, n_jobs=-1),
    param_grid=forest_params,
    cv=5,
    n_jobs=-1,
    scoring='roc_auc'
)
locally_best_forest.fit(train_df, y)

print("Лучшие параметры:", locally_best_forest.best_params_)
print("Лучший ROC-AUC на кросс-валидации:", round(locally_best_forest.best_score_, 3))

Лучшие параметры: {'max_features': np.float64(0.3)}
Лучший ROC-AUC на кросс-валидации: 0.824
CPU times: user 16.9 s, sys: 310 ms, total: 17.2 s
Wall time: 9min 45s


In [71]:
tuned_forest_pred_probs = locally_best_forest.predict_proba(test_df)[:, 1]
write_to_submission_file(tuned_forest_pred_probs, 'credit_scoring_tuned_forest.csv')

Финальный лес - 400 деревьев - 0.83280



In [76]:
%%time
final_forest = RandomForestClassifier(
    n_estimators=400,
    max_features=locally_best_forest.best_params_['max_features'],
    random_state=17,
    n_jobs=-1
)
final_forest.fit(train_df, y)

final_forest_pred = final_forest.predict_proba(test_df)[:, 1]
write_to_submission_file(final_forest_pred, 'credit_scoring_final_forest.csv')

CPU times: user 1min 10s, sys: 334 ms, total: 1min 11s
Wall time: 51.6 s


Нормализация

In [78]:
from sklearn.linear_model import LogisticRegression, LinearRegression
import numpy as np

mean_vals = np.mean(train_df, axis=0)
std_vals = np.std(train_df, axis=0, ddof=0)

# 2. Защита от деления на ноль
epsilon = 1e-8
std_vals = np.where(std_vals == 0, epsilon, std_vals)

# 3. Применяем формулу нормализации: z = (x - mean) / std
X_train_scaled = (train_df - mean_vals) / std_vals

X_test_scaled = (test_df - mean_vals) / std_vals

print(f"Среднее по столбцам после нормализации train: \n{np.mean(X_train_scaled, axis=0).round(5)}")
print(f"Стд. отклонение после нормализации train: \n{np.std(X_train_scaled, axis=0).round(5)}")

Среднее по столбцам после нормализации train: 
DIR                      0.0
Age                      0.0
NumLoans                 0.0
NumRealEstateLoans       0.0
NumDependents            0.0
Num30-59Delinquencies   -0.0
Num60-89Delinquencies    0.0
Income                  -0.0
BalanceToCreditLimit    -0.0
dtype: float64
Стд. отклонение после нормализации train: 
DIR                      1.0
Age                      1.0
NumLoans                 1.0
NumRealEstateLoans       1.0
NumDependents            1.0
Num30-59Delinquencies    1.0
Num60-89Delinquencies    1.0
Income                   1.0
BalanceToCreditLimit     1.0
dtype: float64


Логистическая регрессия - 0.75721


In [79]:
from sklearn.linear_model import LogisticRegression
log_reg = LogisticRegression(
    random_state=17,
    max_iter=1000,
    class_weight='balanced',
    n_jobs=-1
)
log_reg.fit(X_train_scaled, y)

print("✅ Логистическая регрессия обучена")
# Предсказываем вероятности дефолта
log_reg_pred_probs = log_reg.predict_proba(X_test_scaled)[:, 1]

write_to_submission_file(log_reg_pred_probs, 'credit_scoring_logistic_regression.csv')

print("✅ Прогноз сохранен в 'credit_scoring_logistic_regression.csv'")

✅ Логистическая регрессия обучена
✅ Прогноз сохранен в 'credit_scoring_logistic_regression.csv'
